# Trabajo Final HCC

Primero: Descargar la libreria 'pygame'

In [577]:
#!pip install pygame

pygame: Used to display images, text, colors, and to detect user input from the keyboard or mouse.

pygame.gfxdraw: Used to draw shapes (such as circles or lines) with improved visual quality.

sys: Used to close the program when the user requests it.

json: Used to save and load information (such as lists or data) into files, so it is not lost when the application is closed.

Then, I define the screen size, colors, typography, etc.


In [578]:
import pygame
import pygame.gfxdraw
import sys
import json

pygame.init()    # Activate all internal Pygame functions before using it

WIDTH, HEIGHT = 1100, 520
screen = pygame.display.set_mode((WIDTH, HEIGHT))
pygame.display.set_caption("ADHD Organizer")

# Color palette
WHITE = (245, 248, 247)        # Very light off-white
BLACK = (50, 50, 50)           # Very dark gray for text
MINT_GREEN = (26, 188, 156)    # Mint green
LIGHT_GRAY = (230, 230, 230)   # Very light gray for backgrounds
GRAY = (130, 130, 130)         # Medium gray
DARK_GRAY = (100, 100, 100)    # Dark gray for less highlighted buttons
RED = (220, 20, 60)            # Red for alerts
PINK = (240, 128, 128)         # Used for clothing
CREAM = (255, 253, 208)        # Optional background color
GREEN = (144, 190, 109)        # Green for other buttons
LIGHT_GREEN = (230, 255, 230)  # Background for schedule box
SOFT_BLACK = (100, 100, 100)   # Border for schedule box

# Fonts
font = pygame.font.SysFont("Montserrat", 28)
title_font = pygame.font.SysFont("Montserrat-SemiBold.ttf", 36, bold=False)
alert_font = pygame.font.SysFont("Montserrat", 18)
mood_font = pygame.font.SysFont("Montserrat", 20)

# Files
TASKS_FILE = "tasks.json"
CLOTHES_FILE = "clothes.json"
APPLIANCES_FILE = "appliances.json"
INTERESTS_FILE = "interests.json"
JOBS_FILE = "jobs.json"
DOORS_FILE = "doors.json"

# Data
tasks = []
clothes = []
appliances = []
interests = []
jobs = []
doors = []

I define two functions: load_data() and save_data(), which allow the app to save and retrieve information even after it is closed.

load_data() reads JSON files, while save_data() stores the current content of the lists into JSON files.


In [579]:
def load_data():
    global tasks, clothes, appliances, interests, jobs, doors  # These are global variables, not local to the function
    
    for file, data_list in [
        (TASKS_FILE, tasks),
        (CLOTHES_FILE, clothes),
        (APPLIANCES_FILE, appliances),
        (INTERESTS_FILE, interests),
        (JOBS_FILE, jobs),
        (DOORS_FILE, doors)
    ]:
        try:
            with open(file, "r", encoding="utf-8") as f:
                data = json.load(f)
                data_list.clear()
                data_list.extend(data)
        except:
            continue


def save_data():
    for file, data_list in [
        (TASKS_FILE, tasks),
        (CLOTHES_FILE, clothes),
        (APPLIANCES_FILE, appliances),
        (INTERESTS_FILE, interests),
        (JOBS_FILE, jobs),
        (DOORS_FILE, doors)
    ]:
        with open(file, "w", encoding="utf-8") as f:
            json.dump(data_list, f, indent=2, ensure_ascii=False)


I define three classes that allow the creation of reusable buttons. Each button has its own position, size, text, and color.  
The draw() method displays the button with its design, and clicked(pos) detects whether it has been pressed by the user.

Button: used for large buttons with labels such as "Tasks" or "Clothes".

DeleteButton: small buttons with an "X", used to remove items.

HamburgerButton: represented by three horizontal lines, used to open and close a side menu.


In [580]:
class Button:
    def __init__(self, x, y, w, h, text, color=MINT_GREEN):
        self.rect = pygame.Rect(x, y, w, h)
        self.text = text
        self.color = color

    def draw(self, screen):
        pygame.draw.rect(screen, self.color, self.rect, border_radius=10)
        txt = font.render(self.text, True, WHITE)
        txt_rect = txt.get_rect(center=self.rect.center)  # Center the text
        screen.blit(txt, txt_rect)

    def clicked(self, pos):  # pos = mouse position
        return self.rect.collidepoint(pos)  # Returns True if clicked

In [581]:
class DeleteButton:
    def __init__(self, x, y, size=25, color=DARK_GRAY):
        self.rect = pygame.Rect(x, y, size, size)
        self.color = color
        # self.text = "X"
        # self.font = pygame.font.SysFont("Segoe UI", 22, bold=True)

    def draw(self, screen):
        pygame.draw.rect(screen, self.color, self.rect, border_radius=5)
        
        margin = 7
        pygame.draw.line(
            screen,
            WHITE,
            (self.rect.x + margin, self.rect.y + margin),
            (self.rect.x + self.rect.width - margin, self.rect.y + self.rect.height - margin),
            3,
        )
        pygame.draw.line(
            screen,
            WHITE,
            (self.rect.x + self.rect.width - margin, self.rect.y + margin),
            (self.rect.x + margin, self.rect.y + self.rect.height - margin),
            3,
        )

        # txt = self.font.render(self.text, True, WHITE)
        # txt_rect = txt.get_rect(center=self.rect.center)
        # screen.blit(txt, txt_rect)

    def clicked(self, pos):
        return self.rect.collidepoint(pos)


In [582]:
class HamburgerButton:
    def __init__(self, x, y, size=30, color=DARK_GRAY):
        self.rect = pygame.Rect(x, y, size, size)
        self.color = color
        self.open = False

    def draw(self, screen):
        pygame.draw.rect(screen, self.color, self.rect, border_radius=5)
        
        if not self.open:
            # Draw the 3 horizontal lines of the hamburger menu
            for i in range(3):
                pygame.draw.line(
                    screen,
                    WHITE,
                    (self.rect.x + 5, self.rect.y + 7 + i * 8),
                    (self.rect.x + self.rect.width - 5, self.rect.y + 7 + i * 8),
                    3,
                )
        else:
            # Draw an "X" when the menu is open
            margin = 7
            pygame.draw.line(
                screen,
                WHITE,
                (self.rect.x + margin, self.rect.y + margin),
                (self.rect.x + self.rect.width - margin, self.rect.y + self.rect.height - margin),
                3,
            )
            pygame.draw.line(
                screen,
                WHITE,
                (self.rect.x + self.rect.width - margin, self.rect.y + margin),
                (self.rect.x + margin, self.rect.y + self.rect.height - margin),
                3,
            )

    def clicked(self, pos):
        return self.rect.collidepoint(pos)

    def toggle(self):
        self.open = not self.open   # Toggle between open and closed state


Define the function: input_text
    Displays a box where the user can type text in real time.
    It is used to enter new names for tasks, clothing items, appliances, etc.
    It detects keys such as ENTER to accept or BACKSPACE to delete.


In [583]:
def input_text(prompt):
    text = ""
    active = True
    clock = pygame.time.Clock()  # Controls how many times per second the loop runs
    
    while active:
        for event in pygame.event.get():
            if event.type == pygame.QUIT:
                save_data()
                pygame.quit()
                sys.exit()
                
            if event.type == pygame.KEYDOWN:
                if event.key == pygame.K_RETURN:
                    active = False
                elif event.key == pygame.K_BACKSPACE:
                    text = text[:-1]
                else:
                    text += event.unicode  # Returns the pressed key

        screen.fill(WHITE)
        text_render = font.render(prompt + text, True, BLACK)  # The prompt is what appears on screen
        screen.blit(text_render, (50, 200))
        
        pygame.display.flip()
        clock.tick(30)  # Limits to 30 frames per second

    return text

I define a function that draws text on the screen with a horizontal line crossing it out (like when you want to mark something as "completed" or "deleted").

In [584]:
def strikethrough_text(text, font, color, pos, screen):
    rendered_text = font.render(text, True, color)
    screen.blit(rendered_text, pos)
    
    x, y = pos
    width = rendered_text.get_width()
    line_y = y + rendered_text.get_height() // 2
    
    pygame.draw.line(screen, color, (x, line_y), (x + width, line_y), 3)

I define a function that displays a confirmation window, allowing the user to decide whether they want to add a new item or not.

In [585]:
def confirm_add():
    clock = pygame.time.Clock()
    confirm_button = Button(300, 300, 150, 50, "Add", MINT_GREEN)
    cancel_button = Button(500, 300, 150, 50, "Cancel", RED)

    while True:
        screen.fill(WHITE)
        message = font.render("Do you want to add a new item?", True, DARK_GRAY)
        screen.blit(message, (500 - message.get_width() // 2, 200))

        confirm_button.draw(screen)
        cancel_button.draw(screen)

        pygame.display.flip()
        clock.tick(30)

        for event in pygame.event.get():
            if event.type == pygame.QUIT:
                save_data()
                pygame.quit()
                sys.exit()

            elif event.type == pygame.MOUSEBUTTONDOWN and event.button == 1:
                pos = pygame.mouse.get_pos()
                if confirm_button.clicked(pos):
                    return True
                elif cancel_button.clicked(pos):
                    return False

            elif event.type == pygame.KEYDOWN:
                if event.key == pygame.K_ESCAPE:
                    return False

I define a function that allows the user to add a new task with subtasks.

In [586]:
def add_task():
    if confirm_add():
        name = input_text("Task name: ")
        subtasks = []
        
        while True:
            sub = input_text("Subtask (press ENTER empty to finish): ")
            if sub == "":
                break
            subtasks.append({"name": sub, "completed": False})
        
        tasks.append({"name": name, "subtasks": subtasks})
        save_data()

I define a function that opens the screen to edit a specific task.

In [587]:
def edit_task(task_index):
    clock = pygame.time.Clock()
    task = tasks[task_index]
    add_sub_button = Button(600, 450, 160, 40, "+ Subtask", GREEN)
    back_button = Button(770, 450, 80, 40, "Back", RED)
    delete_positions = []

    while True:
        screen.fill(WHITE)
        title = title_font.render(f"Task: {task['name']}", True, DARK_GRAY)
        screen.blit(title, (30, 20))
            
        y = 80
        delete_positions.clear()
        
        if not task['subtasks']:
            notice = font.render("No subtasks.", True, DARK_GRAY)
            screen.blit(notice, (50, y))
        else:
            for j, sub in enumerate(task['subtasks']):
                sub_color = GREEN if sub['completed'] else RED
                status = "√" if sub['completed'] else ""
                sub_text = f"   - {sub['name']} {status}"
                
                if sub['completed']:
                    strikethrough_text(sub_text, font, sub_color, (70, y), screen)
                else:
                    sub_txt_render = font.render(sub_text, True, sub_color)
                    screen.blit(sub_txt_render, (70, y))
                
                delete_button = DeleteButton(650, y - 2)
                delete_button.draw(screen)
                delete_positions.append((delete_button, j))
                y += 30

        add_sub_button.draw(screen)
        back_button.draw(screen)

        pygame.display.flip()

        for event in pygame.event.get():
            if event.type == pygame.QUIT:
                save_data()
                pygame.quit()
                sys.exit()

            elif event.type == pygame.MOUSEBUTTONDOWN:
                pos = pygame.mouse.get_pos()
                
                if event.button == 1:
                    if add_sub_button.clicked(pos):
                        if confirm_add():
                            sub_name = input_text("Subtask name: ")
                            if sub_name.strip():
                                task['subtasks'].append({"name": sub_name, "completed": False})
                                save_data()
                        break

                    elif back_button.clicked(pos):
                        return

                    y_pos = 80
                    for j, sub in enumerate(task['subtasks']):
                        rect_sub = pygame.Rect(70, y_pos, 580, 28)
                        if rect_sub.collidepoint(pos):
                            task['subtasks'][j]['completed'] = not task['subtasks'][j]['completed']
                            save_data()
                            break
                        y_pos += 30

                    for delete_button, j in delete_positions:
                        if delete_button.clicked(pos):
                            task['subtasks'].pop(j)
                            save_data()
                            break

            elif event.type == pygame.KEYDOWN:
                if event.key == pygame.K_ESCAPE:
                    return

I define a class that represents an editable box within a time grid, used to display and edit notes or tasks associated with a specific time.

In [588]:
class ScheduleBox:
    def __init__(self, x, y, width, height, time, column=0):
        self.x = x
        self.y = y
        self.width = width
        self.height = height
        self.time = time
        self.column = column  # 0 = left, 1 = right
        self.text = ""
        self.active = False

    def rect(self, offset_y=0):
        return pygame.Rect(self.x, self.y + offset_y, self.width, self.height)

    def draw(self, screen, font, offset_y=0):
        rect = self.rect(offset_y)
        border_color = SOFT_BLACK
        background_color = WHITE if not self.active else LIGHT_GREEN
        
        pygame.draw.rect(screen, background_color, rect)
        pygame.draw.rect(screen, border_color, rect, 2)

        time_text = font.render(self.time, True, DARK_GRAY)

        time_margin = 80  # Prevent overlapping

        if self.column == 0:
            time_pos_x = rect.x - time_margin
        else:
            time_pos_x = rect.x + rect.width + 10

        text_bg = pygame.Rect(time_pos_x, rect.y + 5, time_text.get_width(), time_text.get_height())
        pygame.draw.rect(screen, WHITE, text_bg)

        screen.blit(time_text, (time_pos_x, rect.y + 5))

        input_text_render = font.render(self.text, True, DARK_GRAY)
        screen.blit(input_text_render, (rect.x + 5, rect.y + 5))

    def handle_event(self, event):
        if event.type == pygame.KEYDOWN and self.active:
            if event.key == pygame.K_BACKSPACE:
                self.text = self.text[:-1]
            elif event.key == pygame.K_RETURN:
                self.active = False
            elif len(self.text) < 40:
                self.text += event.unicode

I define a class that represents an editable cell within a time grid, allowing the user to display and edit notes or tasks associated with a specific time.

From this point on, I create the menus: Tasks, Clothes, Appliances, Interests, Jobs, and Doors.  
Each menu works as a dedicated screen for its specific category.

In [589]:
def menu_tasks():
    clock = pygame.time.Clock()
    subtask_positions = []
    delete_task_positions = []
    delete_sub_positions = []
    
    add_button = Button(650, 450, 60, 40, "+", MINT_GREEN)
    back_button = Button(720, 450, 80, 40, "Back", RED)

    while True:
        screen.fill(WHITE)
        title = title_font.render("Tasks", True, DARK_GRAY)
        screen.blit(title, (30, 20))

        subtask_positions.clear()
        delete_task_positions.clear()
        delete_sub_positions.clear()
        y = 80
        
        if not tasks:
            notice = font.render("No tasks.", True, DARK_GRAY)
            screen.blit(notice, (50, y))
        else:
            for i, task in enumerate(tasks):
                task_txt = font.render(f"{i+1}. {task['name']}", True, DARK_GRAY)
                screen.blit(task_txt, (50, y))
                
                delete_task_button = DeleteButton(720, y + 2)
                delete_task_button.draw(screen)
                delete_task_positions.append((delete_task_button, i))
                
                y += 30
            
                for j, sub in enumerate(task['subtasks']):
                    status = "√" if sub['completed'] else ""
                    sub_color = GREEN if sub['completed'] else RED
                    sub_text = f"   - {sub['name']} {status}"
                    
                    if sub['completed']:
                        strikethrough_text(sub_text, font, sub_color, (70, y), screen)
                    else:
                        sub_txt_render = font.render(sub_text, True, sub_color)
                        screen.blit(sub_txt_render, (70, y))

                    delete_sub_button = DeleteButton(720, y + 2)
                    delete_sub_button.draw(screen)
                    delete_sub_positions.append((delete_sub_button, i, j))

                    rect = pygame.Rect(70, y, 430, 28)
                    subtask_positions.append((rect, i, j))
                    y += 28

        add_button.draw(screen)
        back_button.draw(screen)

        pygame.display.flip()

        for event in pygame.event.get():
            if event.type == pygame.QUIT:
                save_data()
                pygame.quit()
                sys.exit()

            elif event.type == pygame.MOUSEBUTTONDOWN:
                pos = pygame.mouse.get_pos()

                if event.button == 1:
                    if add_button.clicked(pos):
                        if confirm_add():
                            add_task()
                        break

                    elif back_button.clicked(pos):
                        return

                    for rect, task_i, sub_i in subtask_positions:
                        if rect.collidepoint(pos):
                            tasks[task_i]['subtasks'][sub_i]['completed'] = not tasks[task_i]['subtasks'][sub_i]['completed']
                            save_data()
                            break
                    else:
                        y_check = 80
                        for task_i, task in enumerate(tasks):
                            rect_task = pygame.Rect(50, y_check, 450, 30)
                            if rect_task.collidepoint(pos):
                                edit_task(task_i)
                                break
                            y_check += 30 + len(task['subtasks']) * 28

                    for delete_sub_button, task_i, sub_i in delete_sub_positions:
                        if delete_sub_button.clicked(pos):
                            tasks[task_i]['subtasks'].pop(sub_i)
                            if len(tasks[task_i]['subtasks']) == 0:
                                tasks.pop(task_i)
                            save_data()
                            break

                    for delete_task_button, task_i in delete_task_positions:
                        if delete_task_button.clicked(pos):
                            tasks.pop(task_i)
                            save_data()
                            break

In [590]:
def menu_clothes():
    clock = pygame.time.Clock()
    clothes_positions = []
    delete_positions = []
    
    add_button = Button(650, 450, 60, 40, "+", MINT_GREEN)
    back_button = Button(720, 450, 80, 40, "Back", RED)

    while True:
        screen.fill(WHITE)
        title = title_font.render("Clothes", True, DARK_GRAY)
        screen.blit(title, (30, 20))

        clothes_positions.clear()
        delete_positions.clear()
        y = 80
        
        if not clothes:
            notice = font.render("No clothes.", True, DARK_GRAY)
            screen.blit(notice, (50, y))
        else:
            for i, item in enumerate(clothes):
                status_color = MINT_GREEN if item['clean'] else RED
                status_text = "Clean" if item['clean'] else "Dirty"
                
                text = font.render(f"{i+1}. {item['name']}", True, DARK_GRAY)
                screen.blit(text, (50, y))
                
                status_render = font.render(status_text, True, status_color)
                screen.blit(status_render, (250, y))

                delete_button = DeleteButton(400, y + 2)
                delete_button.draw(screen)
                delete_positions.append((delete_button, i))

                rect = pygame.Rect(50, y, 250, 30)
                clothes_positions.append((rect, i))
                y += 35

        add_button.draw(screen)
        back_button.draw(screen)

        pygame.display.flip()
        clock.tick(30)

        for event in pygame.event.get():
            if event.type == pygame.QUIT:
                save_data()
                pygame.quit()
                sys.exit()

            elif event.type == pygame.MOUSEBUTTONDOWN and event.button == 1:
                pos = event.pos

                if add_button.clicked(pos):
                    if confirm_add():
                        name = input_text("Clothing item name: ")
                        if name.strip() != "":
                            clothes.append({"name": name, "clean": True})
                            save_data()

                elif back_button.clicked(pos):
                    return

                else:
                    # Toggle clean/dirty state
                    for rect, item_i in clothes_positions:
                        if rect.collidepoint(pos):
                            clothes[item_i]['clean'] = not clothes[item_i]['clean']
                            save_data()
                            break

                    # Delete item
                    for delete_button, item_i in delete_positions:
                        if delete_button.clicked(pos):
                            clothes.pop(item_i)
                            save_data()
                            break

In [591]:
def menu_appliances():
    clock = pygame.time.Clock()
    appliance_positions = []
    delete_positions = []
    
    add_button = Button(650, 450, 60, 40, "+", MINT_GREEN)
    back_button = Button(720, 450, 80, 40, "Back", RED)

    while True:
        screen.fill(WHITE)
        title = title_font.render("Appliances", True, DARK_GRAY)
        screen.blit(title, (30, 20))

        appliance_positions.clear()
        delete_positions.clear()
        y = 80

        if not appliances:
            notice = font.render("No appliances.", True, DARK_GRAY)
            screen.blit(notice, (50, y))
        else:
            for i, appliance in enumerate(appliances):
                status_color = RED if appliance['plugged'] else MINT_GREEN
                status_text = "Plugged in" if appliance['plugged'] else "Unplugged"
                
                text = font.render(f"{i+1}. {appliance['name']}", True, DARK_GRAY)
                screen.blit(text, (50, y))
                
                status_render = font.render(status_text, True, status_color)
                screen.blit(status_render, (250, y))

                delete_button = DeleteButton(500, y + 2)
                delete_button.draw(screen)
                delete_positions.append((delete_button, i))

                rect = pygame.Rect(50, y, 250, 30)
                appliance_positions.append((rect, i))
                y += 35

        add_button.draw(screen)
        back_button.draw(screen)

        pygame.display.flip()
        clock.tick(30)

        for event in pygame.event.get():
            if event.type == pygame.QUIT:
                save_data()
                pygame.quit()
                sys.exit()

            elif event.type == pygame.MOUSEBUTTONDOWN and event.button == 1:
                pos = event.pos

                if add_button.clicked(pos):
                    if confirm_add():
                        name = input_text("Appliance name: ")
                        if name.strip():
                            appliances.append({"name": name, "plugged": True})
                            save_data()

                elif back_button.clicked(pos):
                    return

                else:
                    # Toggle plugged/unplugged state
                    for rect, i_appliance in appliance_positions:
                        if rect.collidepoint(pos):
                            appliances[i_appliance]['plugged'] = not appliances[i_appliance]['plugged']
                            save_data()
                            break

                    # Delete appliance
                    for delete_button, i_appliance in delete_positions:
                        if delete_button.clicked(pos):
                            appliances.pop(i_appliance)
                            save_data()
                            break

In [592]:
def menu_interests():
    clock = pygame.time.Clock()
    add_button = Button(650, 450, 60, 40, "+", MINT_GREEN)
    back_button = Button(720, 450, 80, 40, "Back", RED)
    delete_positions = []

    while True:
        screen.fill(WHITE)
        title = title_font.render("Things that make you feel good", True, DARK_GRAY)
        screen.blit(title, (50, 40))

        delete_positions.clear()
        y = 100
        
        for i, idea in enumerate(interests):
            txt = font.render("• " + idea, True, DARK_GRAY)
            screen.blit(txt, (70, y))
            
            delete_button = DeleteButton(700, y)
            delete_button.draw(screen)
            delete_positions.append((delete_button, i))
            
            y += 40

        back_button.draw(screen)
        add_button.draw(screen)

        pygame.display.flip()
        clock.tick(30)

        for event in pygame.event.get():
            if event.type == pygame.QUIT:
                save_data()
                pygame.quit()
                sys.exit()

            elif event.type == pygame.MOUSEBUTTONDOWN and event.button == 1:
                pos = event.pos

                if back_button.clicked(pos):
                    return

                if add_button.clicked(pos):
                    if confirm_add():
                        new_item = input_text("Something that makes you feel good: ")
                        if new_item.strip():
                            interests.append(new_item.strip())
                            save_data()

                for delete_button, i in delete_positions:
                    if delete_button.clicked(pos):
                        interests.pop(i)
                        save_data()
                        break

In [593]:
def menu_works():
    boxes = []
    width, height = 320, 40
    start_y = 100
    spacing = 60

    for i in range(24):
        page = 0 if i < 12 else 1
        page_index = i % 12
        column = 0 if page_index < 6 else 1
        row = page_index % 6

        x = 180 + column * (width + 50)
        y = start_y + row * spacing
        time = f"{i:02}:00"
        box = ScheduleBox(x, y, width, height, time, column)
        boxes.append(box)

    if jobs:
        for i, box in enumerate(boxes):
            if i < len(jobs):
                box.text = jobs[i]

    current_page = 0
    active_box = None

    back_button = Button(50, 30, 100, 40, "Back", RED)
    page_button = Button(800, 30, 120, 40, "Next", MINT_GREEN)

    running = True
    while running:
        for event in pygame.event.get():
            if event.type == pygame.QUIT:
                pygame.quit()
                sys.exit()

            elif event.type == pygame.MOUSEBUTTONDOWN:
                pos = pygame.mouse.get_pos()

                if back_button.clicked(pos):
                    jobs.clear()
                    for box in boxes:
                        jobs.append(box.text)
                    save_data()
                    running = False

                elif page_button.clicked(pos):
                    current_page = 1 - current_page
                    active_box = None

                else:
                    active_box = None
                    start = current_page * 12
                    end = start + 12
                    for i in range(start, end):
                        box = boxes[i]
                        if box.rect(0).collidepoint(pos):
                            box.active = True
                            active_box = box
                        else:
                            box.active = False

            elif event.type == pygame.KEYDOWN and active_box:
                active_box.handle_event(event)

        screen.fill(WHITE)

        start = current_page * 12
        end = start + 12

        for i in range(start, end):
            boxes[i].draw(screen, font, 0)

        back_button.draw(screen)
        page_button.text = "Next" if current_page == 0 else "Previous"
        page_button.draw(screen)

        pygame.display.flip()

In [594]:
def menu_doors():
    clock = pygame.time.Clock()
    door_positions = []
    delete_positions = []
    
    add_button = Button(650, 450, 60, 40, "+", MINT_GREEN)
    back_button = Button(720, 450, 80, 40, "Back", RED)

    while True:
        screen.fill(WHITE)
        title = title_font.render("Doors", True, DARK_GRAY)
        screen.blit(title, (30, 20))

        door_positions.clear()
        delete_positions.clear()
        y = 80
        
        if not doors:
            notice = font.render("No doors.", True, DARK_GRAY)
            screen.blit(notice, (50, y))
        else:
            for i, door in enumerate(doors):
                status_color = MINT_GREEN if door['closed'] else RED
                status_text = "Closed" if door['closed'] else "Open"
                
                text = font.render(f"{i+1}. {door['name']}", True, DARK_GRAY)
                screen.blit(text, (50, y))
                
                status_render = font.render(status_text, True, status_color)
                screen.blit(status_render, (250, y))

                delete_button = DeleteButton(400, y + 2)
                delete_button.draw(screen)
                delete_positions.append((delete_button, i))

                rect = pygame.Rect(50, y, 250, 30)
                door_positions.append((rect, i))
                y += 35

        add_button.draw(screen)
        back_button.draw(screen)

        pygame.display.flip()
        clock.tick(30)

        for event in pygame.event.get():
            if event.type == pygame.QUIT:
                save_data()
                pygame.quit()
                sys.exit()

            elif event.type == pygame.MOUSEBUTTONDOWN and event.button == 1:
                pos = event.pos

                if add_button.clicked(pos):
                    if confirm_add():
                        name = input_text("Door name: ")
                        if name.strip() != "":
                            doors.append({"name": name, "closed": False})
                            save_data()

                elif back_button.clicked(pos):
                    return

                else:
                    # Toggle open/closed state
                    for rect, door_i in door_positions:
                        if rect.collidepoint(pos):
                            doors[door_i]['closed'] = not doors[door_i]['closed']
                            save_data()
                            break

                    # Delete door
                    for delete_button, door_i in delete_positions:
                        if delete_button.clicked(pos):
                            doors.pop(door_i)
                            save_data()
                            break

I define a function that represents a mood bar.

Considering that RED = (255, 0, 0) and GREEN = (0, 255, 0), the following is defined:

RED[0] as the red component of the red color (255)  
RED[1] as the green component of the red color (0)  
RED[2] as the blue component of the red color (0)  

GREEN[0] as the red component of green (0)  
GREEN[1] as the green component of green (255)  
GREEN[2] as the blue component of green (0)  

In [595]:
def draw_mood_bar(screen, x, y, width, height, value):
    # value: from 0 (red) to 100 (green)
    for i in range(width):
        # Calculate proportion of i within width to interpolate color
        proportion = i / (width - 1)
        
        # Interpolate RGB between red and mint green
        r = int(RED[0] * (1 - proportion) + MINT_GREEN[0] * proportion)
        g = int(RED[1] * (1 - proportion) + MINT_GREEN[1] * proportion)
        b = int(RED[2] * (1 - proportion) + MINT_GREEN[2] * proportion)
        color = (r, g, b)
        
        # Draw a small vertical line (1px width)
        pygame.draw.line(screen, color, (x + i, y), (x + i, y + height))
        
    # Handle position
    handle_radius = 15
    handle_x = x + int((value / 100) * width)
    handle_y = y + height // 2

    def lighten_color(color, increase=30):
        r = min(color[0] + increase, 255)
        g = min(color[1] + increase, 255)
        b = min(color[2] + increase, 255)
        return (r, g, b)

    # Get background color under the handle for a lighter center effect
    background_color = screen.get_at((handle_x, handle_y))[:3]
    center_color = lighten_color(background_color)

    # Draw outer black circle
    pygame.gfxdraw.aacircle(screen, handle_x, handle_y, handle_radius, BLACK)
    pygame.gfxdraw.filled_circle(screen, handle_x, handle_y, handle_radius, BLACK)
    
    # Draw inner lighter circle
    pygame.gfxdraw.aacircle(screen, handle_x, handle_y, handle_radius - 4, center_color)
    pygame.gfxdraw.filled_circle(screen, handle_x, handle_y, handle_radius - 4, center_color)    
    
    return handle_x, handle_y, handle_radius

I create animations to display the text that appears on the screen: one comes in from the left, and another comes in from the top.

In [596]:
class AlertAnimation:
    def __init__(self, text, category, y):
        self.text = text
        self.category = category
        self.y = y
        self.x = -300
        self.speed = 15

    def update(self):
        final_position = 50
        if self.x < final_position:
            self.x += self.speed
            if self.x > final_position:
                self.x = final_position  # Stop it if it passes 50

    def draw(self, screen, font):
        alert_text = font.render(self.text, True, BLACK)
        alert_rect = alert_text.get_rect(topleft=(self.x, self.y))
        margin = 8
        background_rect = pygame.Rect(
            alert_rect.left - margin,
            alert_rect.top - margin // 2,
            alert_rect.width + 2 * margin,
            alert_rect.height + margin
        )
        pygame.draw.rect(screen, (220, 220, 220), background_rect, border_radius=8)
        screen.blit(alert_text, alert_rect)
        return alert_rect


In [597]:
class InterestAnimation:
    def __init__(self, text, x_final, y_final):
        self.text = text
        self.x = x_final
        self.y = y_final - 200  # Starts 200 pixels above
        self.y_final = y_final
        self.speed = 15  # Vertical falling speed

    def update(self):
        if self.y < self.y_final:
            self.y += self.speed
            if self.y > self.y_final:
                self.y = self.y_final

    def draw(self, screen, font):
        rendered_text = font.render(self.text, True, DARK_GRAY)
        rect = rendered_text.get_rect(topleft=(self.x, int(self.y)))
        screen.blit(rendered_text, rect)
        return rect

Define a function that allows me to obtain alerts so that they can later appear in the main menu

In [598]:
def get_alerts():
    alerts = []

    for task in tasks:
        for subtask in task.get('subtasks', []):
            if not subtask['completed']:
                alerts.append(("You have pending subtasks", "tasks"))
                break
        else:
            continue
        break

    for clothing in clothes:
        if not clothing['clean']:
            alerts.append(("You have dirty clothes", "clothes"))
            break

    for appliance in appliances:
        if appliance['plugged']:
            alerts.append(("You have appliances plugged in", "appliances"))
            break

    for door in doors:
        if not door["closed"]:
            alerts.append(("Open door: " + door["name"], "Doors"))
            break

    return alerts

Create the main menu where everything is displayed on screen

In [599]:
def main_menu():
    clock = pygame.time.Clock()

    tasks_button = Button(50, 200, 200, 60, "Tasks", MINT_GREEN)
    clothes_button = Button(270, 200, 200, 60, "Clothes", MINT_GREEN)
    appliances_button = Button(490, 200, 250, 60, "Appliances", MINT_GREEN)
    exit_button = Button(800, 450, 150, 50, "Exit", (192, 57, 43))
    work_button = Button(270, 300, 200, 60, "Works", MINT_GREEN)
    doors_button = Button(50, 300, 200, 60, "Doors", MINT_GREEN)
    interests_button = Button(490, 300, 250, 60, "Interests", MINT_GREEN)

    hamburger_button = HamburgerButton(20, 20)
    menu_open = False

    options = [
        ("Tasks", menu_tasks),
        ("Clothes", menu_clothes),
        ("Appliances", menu_appliances),
        ("Works", menu_works),
        ("Doors", menu_doors),
        ("Interests", menu_interests)
    ]

    interests_shown = []

    option_rects = []

    bar_x, bar_y = 50, 130
    bar_w, bar_h = 690, 30
    handle_radius = 15
    mood_state = 0.77
    dragging = False

    animations = {}
    help_animation = None
    interest_animations = None

    while True:
        screen.fill(WHITE)
        load_data()

        title = title_font.render("Organizer", True, DARK_GRAY)
        screen.blit(title, (WIDTH // 2 - title.get_width() // 2, 60))

        value_mood = int(mood_state * 100)
        handle_x, handle_y, handle_radius = draw_mood_bar(
            screen, bar_x, bar_y, bar_w, bar_h, value_mood
        )

        if mood_state < 0.33:
            mood_text = "Bad"
        elif mood_state < 0.66:
            mood_text = "Regular"
        else:
            mood_text = "Good"

        if mood_text in ["Bad", "Regular"]:
            box_x, box_y = 750, 120
            box_w = 275
            box_h = 30 + len(interests[:8]) * 25 + 10

            pygame.draw.rect(
                screen, (230, 230, 230),
                (box_x, box_y, box_w, box_h),
                border_radius=12
            )

            if help_animation is None or interests[:8] != interests_shown:
                help_animation = InterestAnimation(
                    "Would you like to do something helpful?",
                    box_x + 15,
                    box_y + 10
                )

                interest_animations = []
                y_interest = box_y + 40

                for interest in interests[:8]:
                    interest_animations.append(
                        InterestAnimation(interest, box_x + 20, y_interest)
                    )
                    y_interest += 25

                interests_shown = interests[:8].copy()

            help_animation.update()
            help_animation.draw(screen, mood_font)

            for anim in interest_animations:
                anim.update()
                anim.draw(screen, mood_font)
        else:
            help_animation = None
            interest_animations = None

        mood_label = font.render(f"Mood: {mood_text}", True, GRAY)
        screen.blit(mood_label, (bar_x, bar_y - 40))

        for button in [
            tasks_button, clothes_button, appliances_button,
            work_button, doors_button, interests_button, exit_button
        ]:
            button.draw(screen)

        new_alerts = get_alerts()
        new_animations = {}
        alert_positions = []
        y_alert = 400

        for text, category in new_alerts:
            if (text, category) not in animations:
                animations[(text, category)] = AlertAnimation(text, category, y_alert)

            anim = animations[(text, category)]
            anim.y = y_alert
            anim.update()

            rect = anim.draw(screen, alert_font)
            alert_positions.append((rect, category))

            new_animations[(text, category)] = anim
            y_alert += 30

        animations = new_animations

        if hamburger_button.open:
            pygame.draw.rect(screen, (200, 200, 200), (0, 0, 200, HEIGHT))
            option_rects.clear()

            for i, (name, function) in enumerate(options):
                txt = font.render(name, True, DARK_GRAY)
                screen.blit(txt, (20, 65 + i * 50))

                rect = pygame.Rect(20, 65 + i * 50, 160, 30)
                option_rects.append((rect, function))

        hamburger_button.draw(screen)
        pygame.display.flip()
        clock.tick(30)

        for event in pygame.event.get():
            if event.type == pygame.QUIT:
                save_data()
                pygame.quit()
                sys.exit()

            elif event.type == pygame.MOUSEBUTTONDOWN:
                pos = pygame.mouse.get_pos()

                if event.button == 1:
                    dx = pos[0] - handle_x
                    dy = pos[1] - handle_y

                    if dx * dx + dy * dy <= handle_radius * handle_radius:
                        dragging = True

                    elif hamburger_button.clicked(pos):
                        hamburger_button.toggle()

                    elif hamburger_button.open:
                        for rect, function in option_rects:
                            if rect.collidepoint(pos):
                                function()
                                hamburger_button.open = False
                                break

                    else:
                        if tasks_button.clicked(pos):
                            menu_tasks()
                        elif clothes_button.clicked(pos):
                            menu_clothes()
                        elif appliances_button.clicked(pos):
                            menu_appliances()
                        elif work_button.clicked(pos):
                            menu_works()
                        elif doors_button.clicked(pos):
                            menu_doors()
                        elif interests_button.clicked(pos):
                            menu_interests()
                        elif exit_button.clicked(pos):
                            save_data()
                            pygame.quit()
                            sys.exit()
                        else:
                            alert_map = {
                                "tasks": menu_tasks,
                                "clothes": menu_clothes,
                                "appliances": menu_appliances,
                                "work": menu_works,
                                "Doors": menu_doors,
                                "interests": menu_interests
                            }

                            for rect, category in alert_positions:
                                if rect.collidepoint(pos):
                                    function = alert_map.get(category)
                                    if function:
                                        function()
                                    break

            elif event.type == pygame.MOUSEMOTION and dragging:
                pos_x = max(bar_x, min(bar_x + bar_w, event.pos[0]))
                mood_state = (pos_x - bar_x) / bar_w

            elif event.type == pygame.MOUSEBUTTONUP:
                if event.button == 1:
                    dragging = False
   

Define the main function: it reads and loads the saved data (tasks, clothes, appliances, etc.) from files to have the information ready when the app starts, and it calls the main menu, which is the main interface with all buttons and navigation.

In [600]:
def main():
    load_data()
    main_menu()

if __name__ == "__main__":
    main()

SystemExit: 